# Various Scripts to Visualize Image Mask Related Things

### Visualize mask based on provided coordinates

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

# Define mask coordinates
mask_data = {
    "umift_mask_pts": [

        (51, 200), 
        (68, 165), 
        (185, 165), 
        (212, 198)
    ]
    ,
    "resolution": [224, 224]
}

# Create a blank image
resolution = mask_data["resolution"]
mask_image = np.zeros((resolution[0], resolution[1], 3), dtype=np.uint8)

# Convert points to NumPy array for OpenCV
gripper_pts = np.array(mask_data["umift_mask_pts"], np.int32).reshape((-1, 1, 2))

# Draw the masks
cv2.fillPoly(mask_image, [gripper_pts], (0, 255, 0))  # Green mask for gripper

# Display the mask
plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(mask_image, cv2.COLOR_BGR2RGB))
plt.title("Visualized Mask")
plt.axis("on")
plt.show()


### Read a dataset file, apply mask to an image

In [ ]:
import zarr
import os
import matplotlib.pyplot as plt
import json
import cv2
import numpy as np

dataset_path = "/store/real/chuerpan/umift_playback/horizon_0"
mask_path = "/local/real/hjchoi92/repo/PyriteUtility/PyriteUtility/data_pipeline/umift_mask.json"

data = zarr.open(dataset_path, mode='r')
# print(data.tree())

with open(mask_path, "r") as f:
    mask_data = json.load(f)

mask_pts = np.array(mask_data["umift_mask_inference"], dtype=np.int32)  # Extract mask points
mask_resolution = tuple(mask_data["mask_inference_resolution"])  # (H, W)

images = data["obs"]["rgb_0"][:]  # Load all images
image = images[0].copy()

mask = np.ones(image.shape, dtype=np.uint8) * 255
cv2.fillPoly(mask, [mask_pts], (0, 0, 0))
masked_image = cv2.bitwise_and(image, mask)

# print(images.shape)  # (N, H, W, C)

plt.imshow(masked_image)  # Show the first image
plt.show()